In [ ]:
import csv
import json
import re


def extract_metric(value):
    match = re.search(r'\((.*?)\)', value)
    if match:
        return match.group(1)
    else:
        return None

def replace_dot_with_underscore(s):
    return s.replace('.', '_')

def flatten_dict(data_dict):

    max_type_count = 0
    max_abilities_count = 0
    max_evolution_count = 0

    type_count = {}
    abilities_count = {}
    evolution_count = {}

    for key in data_dict:
        type_count[key] = len(data_dict[key]['type'])
        abilities_count[key] = len(data_dict[key]['abilities'])
        evolution_count[key] = len(data_dict[key]['evolution'])

        if type_count[key] > max_type_count:
            max_type_count=type_count[key]
        if abilities_count[key] > max_abilities_count:
            max_abilities_count=abilities_count[key]
        if evolution_count[key] > max_evolution_count:
            max_evolution_count=evolution_count[key]

    out_dict = {}

    for pokemon in data_dict:
        data = data_dict[pokemon]
        out_dict[pokemon] = {}
        for key in data:
            if key == 'type':
                i = 0
                for in_key in range(len(data[key])):
                    out_dict[pokemon][key + "." + str(i)] = data[key][in_key]
                    i += 1
                while i < max_type_count:
                    out_dict[pokemon][key + "." + str(i)] = None
                    i += 1
            if key == 'abilities':
                i = 0
                for in_key in range(len(data[key])):
                    out_dict[pokemon][key + "." + str(i)] = data[key][in_key]
                    i += 1
                while i < max_abilities_count:
                    out_dict[pokemon][key + "." + str(i)] = None
                    i += 1
            if key == 'evolution':
                i = 0
                for in_key in range(len(data[key])):
                    out_dict[pokemon][key + "." + str(i)] = data[key][in_key]
                    i += 1
                while i < max_evolution_count:
                    out_dict[pokemon][key + "." + str(i)] = None
                    i += 1


            if isinstance(data[key], dict):
                for in_key in data[key]:
                    new_in_key = replace_dot_with_underscore(in_key)
                    out_dict[pokemon][key + "." + new_in_key] = data[key][in_key]
            else:
                if key == 'height':
                    height = data.get("height")
                    height = extract_metric(height)
                    out_dict[pokemon][key] = height

                if key == 'weight':
                    weight = data.get("weight")
                    weight = extract_metric(weight)
                    out_dict[pokemon][key] = weight

                if key != 'weight' and key != 'height' and key != 'type':
                    out_dict[pokemon][key] = data[key]

    return out_dict

def json_to_tsv(path):
    with open(path, encoding='utf-8') as json_file:
        data = json.load(json_file)

    if isinstance(data, list):
        data = {item['name']: item for item in data}  # Use 'name' as key


    data = flatten_dict(data)
    fields = list(next(iter(data.values())).keys())

    with open('output_pokemons.tsv', "w", newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file, delimiter='\t', quotechar='|', quoting=csv.QUOTE_MINIMAL)
        writer.writerow(['name'] + fields)

        for pokemon in sorted(data):
            row = [pokemon]
            for field in fields:
                if (field == 'name'):
                    continue
                row.append(data[pokemon].get(field, ""))
            writer.writerow(row)


if __name__ == "__main__":
    json_to_tsv('pokedex.json')